In [ ]:
# Imports
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
# Globais
DADOS_CRIMES = r".\dadosPlanilha\BancoVDE 2023.xlsx"
DADOS_POP = r"./dadosPlanilha/ibge/br_ibge_populacao_municipio.csv"
DADOS_ID = r"./dadosPlanilha/ibge/RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods"
ANO = 2023

In [ ]:
df_popAnual = pd.read_csv(DADOS_POP)
df_popAnual.head(3)

In [ ]:
df_ids = pd.read_excel(DADOS_ID)
df_ids.head(3)

In [ ]:
df_ids = df_ids.drop([
    'Região Geográfica Intermediária',
    'Nome Região Geográfica Intermediária',
    'Região Geográfica Imediata',
    'Nome Região Geográfica Imediata',
    'Município',
    'Unnamed: 9'
    ], axis= 1)
df_ids.head(1)

In [ ]:
import pandas as pd

dicionario_estados = {
    'Acre': 'AC',
    'Alagoas': 'AL',
    'Amapá': 'AP',
    'Amazonas': 'AM',
    'Bahia': 'BA',
    'Ceará': 'CE',
    'Distrito Federal': 'DF',
    'Espírito Santo': 'ES',
    'Goiás': 'GO',
    'Maranhão': 'MA',
    'Mato Grosso': 'MT',
    'Mato Grosso do Sul': 'MS',
    'Minas Gerais': 'MG',
    'Pará': 'PA',
    'Paraíba': 'PB',
    'Paraná': 'PR',
    'Pernambuco': 'PE',
    'Piauí': 'PI',
    'Rio de Janeiro': 'RJ',
    'Rio Grande do Norte': 'RN',
    'Rio Grande do Sul': 'RS',
    'Rondônia': 'RO',
    'Roraima': 'RR',
    'Santa Catarina': 'SC',
    'São Paulo': 'SP',
    'Sergipe': 'SE',
    'Tocantins': 'TO'
}

df_ids['UF_Sigla'] = df_ids['Nome_UF'].map(dicionario_estados)
df_ids.head(10)

In [ ]:
df_ids = df_ids.drop(['UF','Nome_UF'], axis=1)
df_ids.head(3)

In [ ]:
np.unique(df_ids['UF_Sigla'])

In [ ]:
df_completo = pd.read_excel(DADOS_CRIMES)
df_completo.head()

In [ ]:
df = df_completo

In [ ]:
df = df[~df['evento'].isin(['Morte de Agente do Estado', 'Suicídio de Agente do Estado'])].drop(['agente'], axis=1)
df.head()

In [ ]:
df = df[~df['evento'].isin(['Arma de Fogo Apreendida'])].drop(['arma'], axis=1)
df.head()

In [ ]:
df = df[~df['evento'].isin(['Pessoa Desaparecida', 'Pessoa Localizada'])].drop(['faixa_etaria'], axis=1)
df.head()

In [ ]:
df = df[~df['evento'].isin(['Apreensão de Cocaína', 'Apreensão de Maconha'])].drop(['total_peso'], axis=1)
df.head()

In [ ]:
df = df[~df['evento'].isin([
    'Atendimento pré-hospitalar',
    'Busca e salvamento',
    'Combate a incêndios',
    'Emissão de Alvarás de licença',
    'Mandado de prisão cumprido',
    'Mortes a esclarecer (sem indício de crime)',
    'Realização de vistorias'
])]
df.head()

In [ ]:
df = df[~df['municipio'].isin(['NÃO INFORMADO'])]
df.head()

In [ ]:
eventos_alvo = [
    'Furto de veículo', 
    'Roubo a instituição financeira', 
    'Roubo de carga', 
    'Roubo de veículo', 
    'Tráfico de drogas'
]

df_alvo = df[df['evento'].isin(eventos_alvo)]
df_outros = df[~df['evento'].isin(eventos_alvo)]

colunas_chave = ['uf', 'municipio', 'evento', 'data_referencia']
colunas_soma = ['feminino', 'masculino', 'nao_informado', 'total_vitima', 'total']

df_consolidado = df_alvo.groupby(colunas_chave, as_index=False, dropna=False)[colunas_soma].sum(min_count=1)

df_consolidado['abrangencia'] = 'combo'
df = pd.concat([df_outros, df_consolidado], ignore_index=True)

df

In [ ]:
df = df.drop(['total'], axis=1)
df.head(1)

In [ ]:
df = df.drop(['feminino'], axis=1).drop(['masculino'], axis=1).drop(['nao_informado'], axis=1)
df.head(10)

In [ ]:
df['data_referencia'] = pd.to_datetime(df['data_referencia'])

df['ano'] = df['data_referencia'].dt.year
df['mes'] = df['data_referencia'].dt.month
df = df.drop(['data_referencia'], axis=1)

df.head(1)

In [ ]:
def padronizar_texto(coluna):
    return coluna.str.normalize('NFKD') \
                 .str.encode('ascii', errors='ignore') \
                 .str.decode('utf-8') \
                 .str.upper() \
                 .str.strip() 

df['chave_mun'] = padronizar_texto(df['municipio'])
df['chave_uf'] = padronizar_texto(df['uf'])

df_ids['chave_mun'] = padronizar_texto(df_ids['Nome_Município'])
df_ids['chave_uf'] = padronizar_texto(df_ids['UF_Sigla'])

df = df.merge(
    df_ids[['Código Município Completo', 'chave_mun', 'chave_uf']], 
    on=['chave_uf', 'chave_mun'],
    how='left'
)

df = df.rename(columns={'Código Município Completo': 'Cod_municipio'})
df = df.drop(columns=['chave_mun', 'chave_uf'])
df.head()

In [ ]:
df = df.merge(
    df_popAnual[['id_municipio', 'ano', 'populacao']],
    left_on=['Cod_municipio', 'ano'],
    right_on=['id_municipio', 'ano'],
    how='left'
)

df = df.drop(columns=['id_municipio'])
df.head()

In [ ]:
df_populacao_unica = df.drop_duplicates(subset=['uf', 'Cod_municipio', 'ano', 'populacao'])

df_populacao_estado = df_populacao_unica.groupby(['uf', 'ano'])['populacao'].sum().reset_index(name='soma_populacao')
df_vitimas_estado = df.groupby(['uf', 'evento', 'ano', 'mes'])['total_vitima'].sum().reset_index(name='soma_total_vitimas')

df_estado = pd.merge(df_vitimas_estado, df_populacao_estado, on=['uf', 'ano'], how='left')

df_estado = df_estado[['uf', 'evento', 'soma_total_vitimas', 'ano', 'mes', 'soma_populacao']]

df_estado.head()

In [ ]:
df_estado['vitimas_escala'] = (df_estado['soma_total_vitimas'] / df_estado['soma_populacao']) * 100000
df_estado.head()

In [ ]:
df['vitimas_escala'] = (df['total_vitima'] / df['populacao']) * 100000
df.head()

In [ ]:
mapeamento_crimes = {
    'Feminicídio': 1,
    'Homicídio doloso': 2,
    'Lesão corporal seguida de morte': 3,
    'Morte no trânsito ou em decorrência dele (exceto homicídio doloso)': 4,
    'Mortes no trânsito': 5,
    'Roubo seguido de morte (latrocínio)': 6,
    'Suicídio': 7,
    'Tentativa de feminicídio': 8,
    'Tentativa de homicídio': 9
}

df['crime_enum'] = df['evento'].map(mapeamento_crimes)
print(df[['evento', 'crime_enum']].head())

In [ ]:
df_estado['crime_enum'] = df_estado['evento'].map(mapeamento_crimes)
print(df[['evento', 'crime_enum']].head())

In [ ]:
df.head()

In [ ]:
df.drop(['evento','abrangencia','Cod_municipio','populacao'], axis=1).to_csv(
    r'./dadosTratados/dados_crimes_2023.csv', 
    index=False
)

In [ ]:
df_estado.head()

In [ ]:
df_estado.drop(['evento','soma_populacao'], axis=1).to_csv(
    r'./dadosTratados/dados_crimes_estados_2023.csv', 
    index=False
)